In [1]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch

## preprocess and embedding

In [4]:
from src.preprocessing import pp
from sklearn.model_selection import train_test_split
import scvi

In [5]:
control_key = "is_control"
condition_keys = "cytokine"
control_name = "PBS"
condition_rep_keys = "gene_embeddings"
random_seed = 42
dataset_name = "PBMC"

In [6]:
filePath = './data/raw/Parse_10M_PBMC_cytokines_Donor1_10per.h5ad'
adata = sc.read_h5ad(filePath)
# adata = adata[adata.obs.sample(frac=0.5, random_state=42).index].to_memory()
print(adata)

AnnData object with n_obs × n_vars = 126769 × 40352
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type'
    var: 'n_cells'


In [7]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=True)

/home/pqw/anaconda3/envs/virtual_cell/lib/python3.10/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [8]:
gene_fix_map = {
    # CSF 家族
    'M-CSF': 'CSF1',
    'G-CSF': 'CSF3',
    'GM-CSF': 'CSF2',
    
    # 干扰素 (IFN) 家族
    'IFN-gamma': 'IFNG',
    'IFN-alpha1': 'IFNA1',
    'IFN-beta': 'IFNB1',
    'IFN-epsilon': 'IFNE',
    'IFN-omega': 'IFNW1',
    'IFN-lambda1': 'IFNL1',
    'IFN-lambda2': 'IFNL2',
    'IFN-lambda3': 'IFNL3',
    
    # 白介素 (IL) 家族 - 去掉横杠并修正别名
    'IL-1-alpha': 'IL1A',
    'IL-1-beta': 'IL1B',
    'IL-1Ra': 'IL1RN',
    'IL-2': 'IL2',
    'IL-3': 'IL3',
    'IL-4': 'IL4',
    'IL-5': 'IL5',
    'IL-6': 'IL6',
    'IL-7': 'IL7',
    'IL-8': 'CXCL8',  # IL-8 的标准基因名是 CXCL8
    'IL-9': 'IL9',
    'IL-10': 'IL10',
    'IL-11': 'IL11',
    'IL-12': 'IL12A', # 异源二聚体，映射到A亚基
    'IL-13': 'IL13',
    'IL-15': 'IL15',
    'IL-16': 'IL16',
    'IL-17A': 'IL17A',
    'IL-17B': 'IL17B',
    'IL-17C': 'IL17C',
    'IL-17D': 'IL17D',
    'IL-17E': 'IL25', # IL-17E 的标准基因名是 IL25
    'IL-17F': 'IL17F',
    'IL-18': 'IL18',
    'IL-19': 'IL19',
    'IL-20': 'IL20',
    'IL-21': 'IL21',
    'IL-22': 'IL22',
    'IL-23': 'IL23A', # 异源二聚体，映射到A亚基
    'IL-24': 'IL24',
    'IL-26': 'IL26',
    'IL-27': 'IL27',
    'IL-31': 'IL31',
    'IL-32-beta': 'IL32', # beta 是剪接变体，基因还是 IL32
    'IL-33': 'IL33',
    'IL-34': 'IL34',
    'IL-35': 'EBI3',  # 异源二聚体 (IL12A + EBI3)，映射到 EBI3
    'IL-36-alpha': 'IL36A',
    'IL-36Ra': 'IL36RN',
    
    # 肿瘤坏死因子 (TNF) 超家族
    'TNF-alpha': 'TNF',
    'TRAIL': 'TNFSF10',
    'TWEAK': 'TNFSF12',
    'APRIL': 'TNFSF13',
    'BAFF': 'TNFSF13B',
    'LIGHT': 'TNFSF14',
    'TL1A': 'TNFSF15',
    'GITRL': 'TNFSF18',
    '4-1BBL': 'TNFSF9',
    'CD27L': 'CD70',
    'CD30L': 'TNFSF8',
    'CD40L': 'CD40LG',
    'OX40L': 'TNFSF4',
    'FasL': 'FASLG',
    'RANKL': 'TNFSF11',
    
    # 生长因子及其他
    'TGF-beta1': 'TGFB1',
    'FGF-beta': 'FGF2',  # 碱性成纤维细胞生长因子
    'EGF': 'EGF',
    'VEGF': 'VEGFA',
    'HGF': 'HGF',
    'IGF-1': 'IGF1',
    'GDNF': 'GDNF',
    'PSPN': 'PSPN',
    'SCF': 'KITLG',      # 干细胞因子
    'FLT3L': 'FLT3LG',
    'TPO': 'THPO',       # 血小板生成素
    'EPO': 'EPO',
    'CT-1': 'CTF1',      # 心肌营养素-1
    'LIF': 'LIF',
    'OSM': 'OSM',
    'TSLP': 'TSLP',
    'ADSF': 'RETN',      # 抵抗素
    'Leptin': 'LEP',
    'Noggin': 'NOG',
    'Decorin': 'DCN',
    'PRL': 'PRL',
    
    # 补体片段 (片段无独立基因，映射到母体蛋白基因)
    'C3a': 'C3',
    'C5a': 'C5',
    
    # 淋巴毒素复合物
    'LT-alpha2-beta1': 'LTA', # 异源三聚体，映射到主亚基
    'LT-alpha1-beta2': 'LTB'  # 异源三聚体，映射到主亚基
}

In [9]:
adata.obs[condition_keys] = (
    adata.obs[condition_keys]
    .astype(str)                            # 1. 解除 category 限制，转为普通字符串
    .map(lambda x: gene_fix_map.get(x, x))  # 2. 替换：如果在字典里就替换，不在（如 PBS）就保持原样 x
    .astype('category')                     # 3. 重新转回 category 类型，节省内存并加快后续计算
)

In [10]:
adata.obs[control_key] = (adata.obs[condition_keys] == control_name)
gene_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()
print(adata.obs[control_key].value_counts())

is_control
False    117605
True       9164
Name: count, dtype: int64


In [11]:
rng = np.random.default_rng(random_seed) 
test_ratio = 0.1
gene_list = list(gene_list)
zero_shot = False

if not zero_shot:
    # 分层抽样 先验证分布内学习能力
    adata_pert = adata[adata.obs[control_key] == False].copy()
    y = adata_pert.obs[condition_keys].astype(str).values
    idx = np.arange(adata_pert.n_obs)
    train_idx, test_idx = train_test_split(
        idx,
        test_size=test_ratio,
        random_state=random_seed,
        stratify=y 
    )
    adata_train = adata_pert[train_idx].copy()
    adata_test = adata_pert[test_idx].copy()
    adata_control = adata[adata.obs[control_key] == True].copy()
    print(gene_list)
    del adata, adata_pert
else:
    # 按基因分割 zero-shot
    n_test = max(1, int(len(gene_list) * test_ratio))
    test_gene = rng.choice(gene_list, size=n_test, replace=False).tolist()
    print(test_gene)
    train_gene = [g for g in gene_list if g not in test_gene]
    print(train_gene)
    adata_control = adata[adata.obs[control_key]==True].copy() # control的target_gene是non-targeting
    adata_train = adata[adata.obs[condition_keys].isin(train_gene)].copy() 
    adata_test = adata[adata.obs[condition_keys].isin(test_gene)].copy()
    del adata

['CSF1', 'IL2', 'CTF1', 'IFNG', 'C5', 'IL17F', 'HGF', 'IL36RN', 'TNFSF18', 'IL10', 'IL1RN', 'IFNL1', 'IL12A', 'IFNL2', 'RETN', 'IFNE', 'IL17C', 'TNFSF9', 'CSF2', 'IL17D', 'TNFSF10', 'IL4', 'IL34', 'FLT3LG', 'TGFB1', 'TSLP', 'IFNA1', 'TNFSF12', 'LTA', 'EGF', 'FASLG', 'TNFSF4', 'IL33', 'IL21', 'IL31', 'IL9', 'IL5', 'IL26', 'KITLG', 'TNFSF15', 'IL20', 'IL17A', 'IFNB1', 'IL36A', 'IL1B', 'NOG', 'IL22', 'TNFSF8', 'IL1A', 'CD70', 'IL19', 'EBI3', 'IL3', 'IL23A', 'GDNF', 'FGF2', 'TNFSF11', 'TNFSF14', 'IL17B', 'PRL', 'IL7', 'CD40LG', 'TNF', 'PSPN', 'C3', 'CSF3', 'IL32', 'CXCL8', 'IL15', 'IL18', 'IL11', 'LIF', 'IFNL3', 'IL13', 'IFNW1', 'IGF1', 'DCN', 'IL6', 'OSM', 'LEP', 'IL25', 'THPO', 'IL16', 'LTB', 'IL27', 'IL24', 'TNFSF13', 'VEGFA', 'TNFSF13B', 'EPO']


In [12]:
sample_rep = "X_pca" 
#sample_rep = "X_scVI" 
#sample_rep = "X_flatvi"
#sample_rep = "X_scVI_linear"
n_comps = 256
n_hidden = 2048
condition_rep_dict = pd.read_pickle("./data/processed/pbmc_data_target_genes_embedding.pkl")
model_ref = None
model_train = None
model_test = None
scvi_save_path = f"./data/processed/model/scvi_random{random_seed}_ncomps{n_comps}_hidden{n_hidden}"
flatvi_save_path = f"./data/processed/model/flatvi_random{random_seed}_ncomps{n_comps}_hidden{n_hidden}"
load_embedding_model = True
if sample_rep == "X_scVI":
    adata_control.layers["counts"] = adata_control.X.copy()
    adata_train.layers["counts"] = adata_train.X.copy()
    if adata_test is not None:
        adata_test.layers["counts"] = adata_test.X.copy()

    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{scvi_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
        try:
            model_train = scvi.model.SCVI.load(f"{scvi_save_path}_train", adata=adata_train)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_train = None
        if adata_test is not None:
            try:
                model_test = scvi.model.SCVI.load(f"{scvi_save_path}_test", adata=adata_test)
            except (FileNotFoundError, OSError, ValueError) as e:
                model_test = None
elif sample_rep in ["X_flatvi","X_scVI_linear"]:
    adata_control.layers["counts"] = adata_control.X.copy()
    adata_train.layers["counts"] = adata_train.X.copy()
    if adata_test is not None:
        adata_test.layers["counts"] = adata_test.X.copy()

In [13]:
adata_control, adata_train, adata_test, model_ref, model_train, model_test = pp.process_to_embedding( 
    adata_control,
    adata_train,
    adata_test = adata_test,
    sample_rep = sample_rep,
    n_comps = n_comps,
    n_hidden = n_hidden,
    model_ref = model_ref,
    model_train = model_train,
    model_test = model_test,
    # scvi_save_path = scvi_save_path,
    # flatvi_save_path = flatvi_save_path,
    condition_rep_dict = condition_rep_dict,
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    condition_keys=condition_keys
    )
if sample_rep == "X_pca":
    sample_rep_scaled = sample_rep + "_scaled" # 额 别忘了
else:
    sample_rep_scaled = sample_rep

[1.0358198  1.013508   0.99478465 0.9548455  0.93870866 0.98208964
 0.94238013 1.061256   0.9687548  0.99578774 0.9535519  0.96942806
 0.9789544  0.9871787  1.0009458  0.9937909  1.0115997  0.9901378
 0.99636054 0.99931836 0.9753351  0.9755484  0.992228   0.98072904
 1.0071826  0.9892692  0.9958991  0.994749   0.9960698  0.98258203
 0.98010474 0.98623645 0.99841344 0.9968288  0.98318875 0.98609924
 0.9934987  0.98893934 0.98133403 0.97790396 0.9648541  0.9982074
 0.9868169  0.9794861  0.98497385 0.98855424 0.9889442  0.9777385
 0.9875347  0.99487406 0.9908378  0.98745644 1.0089267  0.9849723
 0.99632484 0.9987189  0.9920471  0.9947674  0.98686963 0.9954639
 0.9979401  0.9925661  1.0143012  0.9889772  0.9841581  0.9917142
 0.9820922  1.0067127  0.9826096  0.99452865 0.9863267  0.9947967
 0.9959814  0.98950505 0.98889357 1.0067039  1.0243233  1.0156668
 0.98610514 0.9978571  0.99337095 0.98053914 0.9983742  0.98436594
 0.97850645 0.9864864  0.9938093  0.99368054 0.990824   0.9918845
 1.0

In [14]:
adata_control.uns["normalized_m"] = 1/6
adata_train.uns["normalized_m"] = 1/0.9
adata_test.uns["normalized_m"] = 1/0.1

In [15]:
preprocess_save_path = f"./data/processed/{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}_{sample_rep_scaled}"
adata_control.write_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train.write_h5ad(f"{preprocess_save_path}_train.h5ad")
if adata_test is not None:
    adata_test.write_h5ad(f"{preprocess_save_path}_test.h5ad")

In [16]:
print(adata_control)
print(adata_train)
print(adata_test)

AnnData object with n_obs × n_vars = 9164 × 2000
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'is_control'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg', 'pca', 'normalized_m'
    obsm: 'X_pca', 'X_pca_scaled'
    varm: 'PCs', 'X_mean'
    layers: 'counts'
AnnData object with n_obs × n_vars = 105844 × 2000
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'is_control'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions